# **Spatiotemporal anomaly mapping for land degradation and quarry expansion**

**Usage Instructions**

Clone this repository to your local machine.

Create a directory named data/ in the root folder of the project.

Place your pre-processed .tif files inside the newly created data/ folder.

Launch Jupyter Notebook or JupyterLab from your command line terminal.

Open the notebook (e.g., land_use_anomaly_detection.ipynb) and execute all cells sequentially.

In [ ]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# 1. SPATIAL ALIGNMENT FUNCTION


In [ ]:

def reproject_raster(src_path, ref_profile):
    """
    Reprojects a source raster to match the spatial profile of a reference raster.
    Ensures that all input layers share the exact same CRS, resolution, and dimensions.
    """
    with rasterio.open(src_path) as src:
        src_transform = src.transform
        src_crs = src.crs
        dst_transform = ref_profile['transform']
        dst_crs = ref_profile['crs']
        dst_shape = (ref_profile['height'], ref_profile['width'])

        dst_data = np.empty(dst_shape, dtype=src.read(1).dtype)

        # Bilinear resampling is used to maintain the continuous nature of spectral indices
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_data,
            src_transform=src_transform,
            src_crs=src_crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )

    return dst_data



# 2. DATA LOADING AND PREPROCESSING


In [ ]:


# Paths to the temporal difference rasters (2024 - 2017) representing vegetation changes
file_paths = [
    r"data\NDVI_diff_2024_2017.tif",
    r"data\evi_difference.tif",
    r"data\savi_difference_2024_2017.tif"
]

# Extract the baseline geospatial profile from the first file
with rasterio.open(file_paths[0]) as ref_src:
    ref_profile = ref_src.profile

# Reproject all rasters to align perfectly with the reference
raster_data = []
for file_path in file_paths:
    aligned_data = reproject_raster(file_path, ref_profile)
    raster_data.append(aligned_data)

# Stack the 2D arrays into a single 3D array (height, width, 3 indices)
raster_stack = np.stack(raster_data, axis=-1)
height, width, num_features = raster_stack.shape


# 3. NODATA MASKING (CRITICAL METHODOLOGICAL STEP)


In [ ]:


# Exclude NaN values (e.g., image borders or water bodies) to prevent the
# algorithm from classifying empty pixels as extreme anomalies.
valid_mask = ~np.isnan(raster_stack).any(axis=-1)

# Flatten the spatial data into a 2D matrix strictly for valid pixels
valid_pixels = raster_stack[valid_mask]


# 4. ANOMALY DETECTION (ISOLATION FOREST)


In [ ]:


# Initialize the model to detect land degradation (e.g., quarry expansion)
model = IsolationForest(contamination=0.025, random_state=42)

# Train the model only on the valid pixel matrix
model.fit(valid_pixels)

# Extract continuous anomaly scores using the decision_function.
# Lower scores indicate higher anomaly. We invert the sign (-) so that
# positive peak values clearly represent the targeted land use changes.
anomaly_scores_valid = -model.decision_function(valid_pixels)

# Map the 1D predicted scores back onto the original 2D geographical grid
anomaly_map = np.full((height, width), np.nan, dtype=np.float32)
anomaly_map[valid_mask] = anomaly_scores_valid


# 5. VISUALIZATION AND GEO-TIFF EXPORT


In [ ]:


# Visualize the continuous anomaly probability
plt.figure(figsize=(10, 10))
plt.imshow(anomaly_map, cmap='coolwarm', vmin=-0.2, vmax=0.2)
plt.title('Continuous Anomaly Map (NDVI, EVI, SAVI - 2017 to 2024)', fontsize=15)
plt.colorbar(label='Anomaly Probability Score', shrink=0.8)
plt.tight_layout()
plt.show()

# Update the geospatial profile for exporting a continuous map with explicit NoData
export_profile = ref_profile.copy()
export_profile.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan
)

output_file_path = r"data\anomaly_scores_ndvi_evi_savi_0.025.tif"
with rasterio.open(output_file_path, "w", **export_profile) as dst:
    dst.write(anomaly_map, 1)

print(f"Continuous anomaly map successfully saved to: {output_file_path}")


# 6. MODEL VALIDATION (GROUND TRUTH COMPARISON)


In [ ]:

# Load the binary mask of the quarry (1 = quarry change, 0 = stable land)
ground_truth_path = r"data\raster_verità_a_terra_CAVA.tif"
with rasterio.open(ground_truth_path) as gt_src:
    ground_truth = gt_src.read(1)

# Filter the ground truth to match only the pixels where satellite data is valid
gt_valid = ground_truth[valid_mask]

# Calculate ROC and AUC strictly on continuous statistical probabilities
fpr, tpr, _ = roc_curve(gt_valid, anomaly_scores_valid)
roc_auc = auc(fpr, tpr)

print(f"Model AUC Validation: {roc_auc:.4f}")

# Plot the definitive ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Land Use Anomaly Detection (Quarry Expansion)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()